# `rcrc-engine`: Single-Date Inference & Multi-Date Temporal Disease Progression Demo

This notebook demonstrates how to use **`rcrc`** (`rcrc-engine`) in Google Colab, VS Code, or local Jupyter environments to:
1. Run single-date preprocessing (`preprocess_scene`) with the 3-SD Peak NDVI canopy mask and optional inward edge buffer (`buffer_m=-3.0`).
2. Run pixel-wise RCR probability and binary classification (`predict_scene`) and export Cloud-Optimized GeoTIFFs (COGs).
3. Execute a multi-date temporal progression analysis (`analyze_temporal_stack`) across chronological 8-band PlanetScope scenes (`3B_AnalyticMS_SR_8b_harmonized`), mapping pixel transitions (`0: Unaffected`, `1: New Expansion`, `10: Apparent Recovery`, `11: Persistent Infection`).
4. Export tidy longitudinal tables ready for LMM/GLMM analysis in R (`glmmTMB` / `lme4`) or Python (`statsmodels`).

In [ ]:
# Uncomment to install in Google Colab or a fresh environment:
# !pip install -q -e ..  # from examples/ directory
# !pip install -q matplotlib

import logging
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

import rcrc

logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(name)s | %(message)s")
print(f"rcrc-engine version: {rcrc.__version__}")
print(f"Available models: {rcrc.list_models()}")

## 1. Configure Input Paths (PlanetScope 8-Band SR + Field Boundary)

Update the paths below to point to your local or Google Drive PlanetScope `3B_AnalyticMS_SR_8b_harmonized` GeoTIFFs and field boundary vector file (`.gpkg`, `.shp`, or `.geojson`).

In [ ]:
# Example paths matching the RCRC routine_inspection structure for field BNC1 (2026)
DATA_ROOT = Path("../../rcrc/routine_inspection")  # adjust to your data location
SAT_DIR = DATA_ROOT / "raw_satellite" / "BNC1" / "2026"
BOUNDARY_PATH = DATA_ROOT / "vectors" / "Boun_BNC1.gpkg"

# Peak canopy baseline scene (used to build the 3-SD NDVI canopy mask)
PEAK_NDVI_PATH = next(SAT_DIR.glob("*20260701*.tif*")) if SAT_DIR.exists() else Path("BNC1_20260701_SR_8b.tif")

# Chronological dictionary of target scouting dates
SCENE_DICT = {
    "2026-07-27": next(SAT_DIR.glob("*20260727*.tif*")) if SAT_DIR.exists() else Path("BNC1_20260727_SR_8b.tif"),
    "2026-08-08": next(SAT_DIR.glob("*20260808*.tif*")) if SAT_DIR.exists() else Path("BNC1_20260808_SR_8b.tif"),
    "2026-08-24": next(SAT_DIR.glob("*20260824*.tif*")) if SAT_DIR.exists() else Path("BNC1_20260824_SR_8b.tif"),
}
OUTPUT_DIR = Path("./demo_outputs")
print("Peak NDVI scene:", PEAK_NDVI_PATH)
print("Target dates:", list(SCENE_DICT.keys()))

## 2. Single-Date Preprocessing & Inference

Here we preprocess a single capture date (`2026-08-08`) with `sd_mask=3.0` and an optional inward buffer (`buffer_m=-3.0` m) to remove field-edge mixed pixels, then run `predict_scene()`.

In [ ]:
single_ds = rcrc.preprocess_scene(
    scene_path=SCENE_DICT["2026-08-08"],
    peak_ndvi_path=PEAK_NDVI_PATH,
    boundary_path=BOUNDARY_PATH,
    sd_mask=3.0,
    buffer_m=-3.0,
)

single_summary = rcrc.predict_scene(
    preprocessed_data=single_ds,
    model_version="v3.1",
    output_dir=OUTPUT_DIR / "single_date",
    prefix="BNC1_20260808",
    return_rasters=True,
)

for k, v in single_summary.items():
    if k != "prediction_ds":
        print(f"{k:24s}: {v}")

## 3. Multi-Date Temporal Stack & Pixel Transition Analysis

`analyze_temporal_stack()` aligns all dates onto a unified spatial grid (`rioxarray.reproject_match`), applies a shared seasonal canopy mask, evaluates RCRC probabilities across dates, and computes pixel transitions (`0`, `1`, `10`, `11`).

In [ ]:
temporal_res = rcrc.analyze_temporal_stack(
    scene_dict=SCENE_DICT,
    peak_ndvi_path=PEAK_NDVI_PATH,
    boundary_path=BOUNDARY_PATH,
    model_version="v3.1",
    sd_mask=3.0,
    buffer_m=-3.0,
    field_id="BNC1",
    output_dir=OUTPUT_DIR / "temporal",
)

prog_df = temporal_res["progression_df"]
trans_df = temporal_res["transitions_df"]
stack = temporal_res["temporal_stack"]

print("=== Longitudinal Progression Table (Ready for LMM/GLMM) ===")
display(prog_df[[
    "date_iso", "days_since_start", "total_canopy_ha", "rcr_affected_ha",
    "incidence_pct", "mean_probability", "new_expansion_ha", "persistent_ha", "recovery_ha"
]])

print("\n=== Pairwise Pixel Transitions Summary ===")
display(trans_df)

## 4. Visualize Temporal Probabilities & Start-to-End Transitions

Transition codes:
- `0` (`0 -> 0`): **Unaffected**
- `1` (`0 -> 1`): **New Expansion / Late Onset**
- `10` (`1 -> 0`): **Apparent Recovery / Canopy Regrowth**
- `11` (`1 -> 1`): **Persistent Infection**

In [ ]:
n_dates = stack.sizes["time"]
fig, axes = plt.subplots(1, n_dates + 1, figsize=(4.5 * (n_dates + 1), 4.2), dpi=150)

for i in range(n_dates):
    date_lbl = str(stack["time"].values[i])[:10]
    prob_slice = stack["probability"].isel(time=i).to_numpy()
    im = axes[i].imshow(prob_slice, cmap="YlOrRd", vmin=0.0, vmax=1.0)
    axes[i].set_title(f"{date_lbl} | P(RCR)", fontsize=10, fontweight="bold")
    axes[i].axis("off")
    fig.colorbar(im, ax=axes[i], fraction=0.046, pad=0.04)

# Remap transition codes {0, 1, 10, 11} -> {0, 1, 2, 3} for discrete plotting
trans_raw = stack["transition_start_to_end"].to_numpy().astype(float)
trans_plot = np.full_like(trans_raw, np.nan)
for idx, code in enumerate([0, 1, 10, 11]):
    trans_plot[trans_raw == code] = idx

trans_cmap = ListedColormap(["#D8F3DC", "#F59E0B", "#3B82F6", "#DC2626"])
trans_cmap.set_bad((1.0, 1.0, 1.0, 0.0))
im_t = axes[-1].imshow(trans_plot, cmap=trans_cmap, vmin=-0.5, vmax=3.5, interpolation="nearest")
axes[-1].set_title("Start -> End Transitions", fontsize=10, fontweight="bold")
axes[-1].axis("off")
cbar_t = fig.colorbar(im_t, ax=axes[-1], ticks=[0, 1, 2, 3], fraction=0.046, pad=0.04)
cbar_t.ax.set_yticklabels(["Unaffected (0)", "Expansion (1)", "Recovery (10)", "Persistent (11)"], fontsize=8)

plt.tight_layout()
plt.show()